In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../..")

# Joining data

In [ ]:
import glob
import json
import os
import joblib
from tqdm import tqdm


DATA_ROOT = "../../data"

reddit_path = os.path.join(DATA_ROOT, "reddit/03_prepared_data")
reddit_data_files = glob.glob(reddit_path + "/raw" + "*.joblib")
tg_path = os.path.join(DATA_ROOT, "telegram/03_prepared_data")
tg_data_files = glob.glob(tg_path + "/raw" + "*.joblib")

data_files = [i for i in reddit_data_files + tg_data_files if "mapping" not in i]
generation_results_root = os.path.join(DATA_ROOT, "generation_results_raw")
all_files_output = glob.glob(os.path.join(generation_results_root, "*", "*", "*.jsonl"))

all_files_output

In [ ]:
all_predictions = []
for file in all_files_output:
    print(file)

    parts = file.replace("\\", "/").split("/")
    pipeline = parts[-2]   # "reduced" or "full"
    model    = parts[-3]   # "qwen", "gemma_small"

    # reading file:
    with open(file, "rb") as f:
        lines = f.readlines()
        for line in lines:
            try:
                data = json.loads(line)
                all_predictions.append((
                    pipeline,
                    model,
                    data['response']["body"]['choices'][0]['message']['content'],
                    data["custom_id"]
                ))
            except:
                print("Error in line: ", line)
                continue

In [ ]:
all_predictions[0]

In [ ]:
predictions_df = pd.DataFrame(all_predictions, columns=["pipeline", "model", "generated_text", "merging_id"])
def extract_user_thread(row):
    prefix = f"{row['model']}_{row['pipeline']}_"
    rest = row["merging_id"][len(prefix):]
    parts = rest.split("_", 1)
    user_id   = parts[0]
    thread_id = parts[1] if len(parts) > 1 else ""
    return user_id, thread_id

predictions_df[["user_id", "thread_id"]] = predictions_df.apply(
    extract_user_thread, axis=1, result_type="expand"
)
print(len(predictions_df))
predictions_df.head()

In [ ]:
len(predictions_df) / predictions_df[["model", "pipeline"]].drop_duplicates().shape[0]

In [ ]:
real_data = []
for file in tqdm(data_files):
    data = joblib.load(file)
    for item in data:
        real_data.append((
            item["user_id"],
            item["thread_id"],
            item["platform"],
            item["channel"],
            item["persona"].dict(),
            item["thread"].dict(),
            item["true_message"],
            [i.dict() for i in item["most_similar_threads"]]
            )
        )

real_data = pd.DataFrame(real_data, columns=[
    "user_id",
    "thread_id",
    "platform",
    "channel",
    "persona",
    "thread",
    "real_text",
    "most_similar_threads"
])
real_data

In [ ]:
import csv
full_merged_data = pd.merge(predictions_df, real_data, on=["user_id", "thread_id"], how="left")
assert full_merged_data.isna().mean().sum() == 0

full_merged_data.drop(columns=["merging_id"], inplace=True)
full_merged_data["thread_id"] = full_merged_data["thread"].apply(lambda x: x["id"])
new_columns_order = ['model', 'pipeline', 'real_text', 'generated_text', 'user_id', 'thread_id', 'platform',
       'channel', 'persona', 'thread', 'most_similar_threads']
full_merged_data = full_merged_data[new_columns_order]
assert full_merged_data.isna().mean().sum() == 0

# Processing generated text

In [ ]:
from tqdm import tqdm

def replace_unicode_dashes(text: str) -> str:
    return text.replace('\u2014', '-').replace('\u2013', '-')

def replace_curly_quotes(text: str) -> str:
    replacements = {
        '\u2018': "'", '\u2019': "'",
        '\u201c': '"', '\u201d': '"'
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def contains_long_dashes(text: str) -> bool:
    return '\u2014' in text or '\u2013' in text

def contains_curly_quotes(text: str) -> bool:
    curly_quotes = ['\u2018', '\u2019', '\u201c', '\u201d']
    return any(quote in text for quote in curly_quotes)

def remove_outer_quotes(text: str) -> str:
    if not text:
        return text
    if text.startswith('"') and text.endswith('"'):
        return text[1:-1]
    if text.startswith("'") and text.endswith("'"):
        return text[1:-1]
    return text

In [ ]:
from copy import deepcopy
processed_generated_texts = []
changes_log = []
for i, record in tqdm(full_merged_data.iterrows(), total=full_merged_data.shape[0]):

    original_generated_text = record["generated_text"]
    processed_generated_text = deepcopy(original_generated_text)

    user_messages = [t["previous_comments"][-1]["message_text"] for t in record["most_similar_threads"]]
    if any(contains_long_dashes(msg) for msg in user_messages):
        # If long dashes are found, we are not replacing them
        pass
    else:
        processed_generated_text = replace_unicode_dashes(processed_generated_text)

    # Check if user messages contain curly quotes
    if any(contains_curly_quotes(msg) for msg in user_messages):
        pass
    else:
        processed_generated_text = replace_curly_quotes(processed_generated_text)

    processed_generated_text = remove_outer_quotes(processed_generated_text)
    processed_generated_texts.append(processed_generated_text)
    if original_generated_text != processed_generated_text:
        changes_log.append({
            "original": original_generated_text,
            "processed": processed_generated_text
        })
full_merged_data["generated_text"] = processed_generated_texts

In [ ]:
len(changes_log) / len(full_merged_data)

In [ ]:
full_merged_data["persona"] = full_merged_data["persona"].astype(str)
full_merged_data["thread"] = full_merged_data["thread"].astype(str)
full_merged_data["most_similar_threads"] = full_merged_data["most_similar_threads"].astype(str)
full_merged_data.to_parquet(os.path.join(DATA_ROOT, "full_merged_data.parquet"), index=False)